# 深度学习预备知识总结

本notebook整理了《动手学深度学习》第二章的核心概念和代码，涵盖数据操作、线性代数、微积分、自动微分和概率论。

## 本章结构
1. **数据操作** — 张量的创建、运算、索引、广播
2. **线性代数** — 标量、向量、矩阵、点积、矩阵乘法、范数
3. **微积分** — 导数、偏导数、梯度、链式法则
4. **自动微分** — PyTorch的autograd机制
5. **概率** — 基本概率论、条件概率、贝叶斯定理、期望与方差

## 一、数据操作（Data Manipulation）

深度学习存储和操作数据的主要接口是**张量（tensor）**，即$n$维数组。
与NumPy的`ndarray`类似，但PyTorch的`Tensor`额外支持：
- **GPU加速计算**
- **自动微分（autograd）**

### 1.1 创建张量

```python
import torch
```

In [1]:
import torch

# arange创建连续整数张量
x = torch.arange(12)
print("x =", x)
print("shape:", x.shape)    # 形状
print("numel:", x.numel())  # 元素总数

# reshape改变形状（不改变元素）
X = x.reshape(3, 4)
print("X =\n", X)

x = tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
shape: torch.Size([12])
numel: 12
X =
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


**其他创建方式：**

In [7]:
# 全0、全1、随机数、指定值
print("全0:", torch.zeros((2, 3)))
print("全0:", torch.zeros((2, 3, 4)))
print("全1:", torch.ones((2, 3, 4))) #有一点是表示浮点数
print("标准正态随机:", torch.randn(3, 4))
print("指定值:", torch.tensor([[2, 1, 4, 3], [1, 2, 3, 4]]))

全0: tensor([[0., 0., 0.],
        [0., 0., 0.]])
全0: tensor([[[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],

        [[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]]])
全1: tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]])
标准正态随机: tensor([[-2.1770,  0.6010, -1.1535,  0.4731],
        [ 0.6082,  0.9854,  1.5909,  0.7599],
        [-0.6107, -0.6113, -0.7672, -0.0285]])
指定值: tensor([[2, 1, 4, 3],
        [1, 2, 3, 4]])


### 1.2 运算符

**按元素运算（elementwise）**：标准算术运算符（`+`、`-`、`*`、`/`、`**`）对同形状张量逐元素操作。

**连结（concatenate）**：`torch.cat()`沿指定轴拼接张量。

**逻辑运算**：`X == Y` 返回布尔张量。

**求和**：`X.sum()` 返回单元素张量。

In [9]:
x = torch.tensor([1.0, 2, 4, 8])
y = torch.tensor([2, 2, 2, 2])

# 按元素运算
print("x + y =", x + y)
print("x * y =", x * y)
print("x ** y =", x ** y) #幂运算
print("exp(x) =", torch.exp(x))

# 连结
X = torch.arange(12, dtype=torch.float32).reshape(3, 4)
Y = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])
print("X =\n", X)
print("Y =\n", Y)  
print("沿行连结:\n", torch.cat((X, Y), dim=0))
print("沿列连结:\n", torch.cat((X, Y), dim=1))

# 逻辑运算与求和
print("X == Y:\n", X == Y)
print("X.sum() =", X.sum())

x + y = tensor([ 3.,  4.,  6., 10.])
x * y = tensor([ 2.,  4.,  8., 16.])
x ** y = tensor([ 1.,  4., 16., 64.])
exp(x) = tensor([2.7183e+00, 7.3891e+00, 5.4598e+01, 2.9810e+03])
X =
 tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])
Y =
 tensor([[2., 1., 4., 3.],
        [1., 2., 3., 4.],
        [4., 3., 2., 1.]])
沿行连结:
 tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [ 2.,  1.,  4.,  3.],
        [ 1.,  2.,  3.,  4.],
        [ 4.,  3.,  2.,  1.]])
沿列连结:
 tensor([[ 0.,  1.,  2.,  3.,  2.,  1.,  4.,  3.],
        [ 4.,  5.,  6.,  7.,  1.,  2.,  3.,  4.],
        [ 8.,  9., 10., 11.,  4.,  3.,  2.,  1.]])
X == Y:
 tensor([[False,  True, False,  True],
        [False, False, False, False],
        [False, False, False, False]])
X.sum() = tensor(66.)


### 1.3 广播机制（Broadcasting Mechanism）

当两个张量形状不同时，通过适当复制元素扩展为相同形状，再执行按元素操作。

In [10]:
# 3x1 + 1x2 → 广播为 3x2
a = torch.arange(3).reshape(3, 1)
b = torch.arange(2).reshape(1, 2)
print("a =\n", a)
print("b =\n", b)
print("a + b =\n", a + b)  # a复制列，b复制行

a =
 tensor([[0],
        [1],
        [2]])
b =
 tensor([[0, 1]])
a + b =
 tensor([[0, 1],
        [1, 2],
        [2, 3]])


### 1.4 索引和切片

与Python数组一致：索引从0开始，`-1`表示最后一个，`[1:3]`表示第2到第3个。
可以通过索引读取和写入元素。

In [ ]:
X = torch.arange(12, dtype=torch.float32).reshape(3, 4)
print("X =\n", X)
print("X[-1] =", X[-1])          # 最后一行
print("X[1:3] =\n", X[1:3])      # 第2-3行
X[0:2, :] = 12                   # 批量赋值
X[1, 2] = 9                      # 写入单个元素，实则是第二行第三个元素
print("修改后 X =\n", X)

X =
 tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])
X[-1] = tensor([ 8.,  9., 10., 11.])
X[1:3] =
 tensor([[ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])
修改后 X =
 tensor([[12., 12., 12., 12.],
        [12., 12.,  9., 12.],
        [ 8.,  9., 10., 11.]])


### 1.5 节省内存

`Y = Y + X` 会分配新内存，而非原地操作。原地操作用 `Y[:] = Y + X` 或 `Y += X`。

### 1.6 转换为其他Python对象

- `X.numpy()` — Tensor → NumPy
- `torch.tensor(A)` — NumPy → Tensor
- `a.item()` — 标量Tensor → Python标量

## 二、线性代数（Linear Algebra）

### 2.1 基本数学对象

| 对象 | 英文 | 符号 | 张量维度 | 例子 |
|------|------|------|----------|------|
| 标量 | scalar | $x$ | 0维（单元素） | `torch.tensor(3.0)` |
| 向量 | vector | $\mathbf{x}$ | 1维 | `torch.arange(4)` |
| 矩阵 | matrix | $\mathbf{A}$ | 2维 | `torch.arange(12).reshape(3,4)` |
| 张量 | tensor | $\mathsf{X}$ | 任意维 | `torch.arange(24).reshape(2,3,4)` |

**注意**："维度"在不同语境有不同含义：
- 向量的维度 = 向量的长度（元素个数）
- 张量的维度 = 轴的数量

### 2.2 张量算法基本性质

- **按元素运算**：相同形状的张量，逐元素操作
- **Hadamard积**（$\odot$）：两个矩阵的按元素乘法（`A * B`），注意与矩阵乘法不同
- **标量运算**：标量与张量的加法/乘法，每个元素都与标量运算

In [17]:
A = torch.arange(20, dtype=torch.float32).reshape(5, 4)
B = A.clone()  # 复制
print("A =\n", A)

# Hadamard积（按元素乘法）
print("A * B =\n", A * B)

# 标量运算
a = 2
X = torch.arange(24).reshape(2, 3, 4)
print("(a * X).shape =", (a * X).shape)  # 形状不变

A =
 tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [12., 13., 14., 15.],
        [16., 17., 18., 19.]])
A * B =
 tensor([[  0.,   1.,   4.,   9.],
        [ 16.,  25.,  36.,  49.],
        [ 64.,  81., 100., 121.],
        [144., 169., 196., 225.],
        [256., 289., 324., 361.]])
(a * X).shape = torch.Size([2, 3, 4])


### 2.3 降维（Reduction）

- `A.sum()` — 所有元素求和（标量）
- `A.sum(axis=0)` — 沿行求和（按列压缩），轴0消失
- `A.sum(axis=1)` — 沿列求和（按行压缩），轴1消失
- `A.mean()` / `A.mean(axis=0)` — 平均值
- `A.sum(axis=1, keepdims=True)` — 保持轴数不变（便于广播）
- `A.cumsum(axis=0)` — 累积求和（不降维）

In [18]:
A = torch.arange(20, dtype=torch.float32).reshape(5, 4)
B = A.clone()  # 复制
print("A =\n", A)
print("A.sum() =", A.sum())
print("A.sum(axis=0) =", A.sum(axis=0))  # 沿行求和
print("A.sum(axis=1) =", A.sum(axis=1))  # 沿列求和
print("A.sum(axis=[0, 1]) =", A.sum(axis=[0, 1]))  # 沿行和列求和
print("A.mean() =", A.mean())  # 平均值
print("A.mean(axis=0) =", A.mean(axis=0))  # 沿行求平均值
print("A.mean(axis=1) =", A.mean(axis=1))  # 沿列求平均值
print("A.cumsum(axis=0) =", A.cumsum(axis=0))  # 沿行累积求和
print("A.cumsum(axis=1) =", A.cumsum(axis=1))  # 沿列累积求和

A =
 tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [12., 13., 14., 15.],
        [16., 17., 18., 19.]])
A.sum() = tensor(190.)
A.sum(axis=0) = tensor([40., 45., 50., 55.])
A.sum(axis=1) = tensor([ 6., 22., 38., 54., 70.])
A.sum(axis=[0, 1]) = tensor(190.)
A.mean() = tensor(9.5000)
A.mean(axis=0) = tensor([ 8.,  9., 10., 11.])
A.mean(axis=1) = tensor([ 1.5000,  5.5000,  9.5000, 13.5000, 17.5000])
A.cumsum(axis=0) = tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  6.,  8., 10.],
        [12., 15., 18., 21.],
        [24., 28., 32., 36.],
        [40., 45., 50., 55.]])
A.cumsum(axis=1) = tensor([[ 0.,  1.,  3.,  6.],
        [ 4.,  9., 15., 22.],
        [ 8., 17., 27., 38.],
        [12., 25., 39., 54.],
        [16., 33., 51., 70.]])


### 2.4 点积（Dot Product）

$$\mathbf{x}^\top\mathbf{y} = \sum_{i=1}^{d} x_i y_i$$

应用：加权平均、余弦相似度。

### 2.5 矩阵-向量积（Matrix-Vector Product）

$$\mathbf{A}\mathbf{x} = \begin{bmatrix} \mathbf{a}^\top_{1}\mathbf{x} \\ \vdots \\ \mathbf{a}^\top_{m}\mathbf{x} \end{bmatrix}$$

这里的ai是A的每一行向量再做转置。

将矩阵看作从$\mathbb{R}^n$到$\mathbb{R}^m$的变换。

### 2.6 矩阵-矩阵乘法（Matrix-Matrix Multiplication）

$$\mathbf{C} = \mathbf{AB}, \quad c_{ij} = \mathbf{a}^\top_i \mathbf{b}_j$$

即$A$的第$i$行与$B$的第$j$列的点积。注意与Hadamard积区分。

In [21]:
x = torch.arange(4, dtype=torch.float32)
y = torch.ones(4, dtype=torch.float32)
print("x =", x)
print("y =", y)
# 点积
print("x · y =", torch.dot(x, y))       # 也可以用 torch.sum(x * y)

# 矩阵-向量积
A = torch.arange(20, dtype=torch.float32).reshape(5, 4)
print("A =\n", A)
print("Ax =", torch.mv(A, x))

# 矩阵-矩阵乘法
B = torch.ones(4, 3)
print("AB =\n", torch.mm(A, B))

x = tensor([0., 1., 2., 3.])
y = tensor([1., 1., 1., 1.])
x · y = tensor(6.)
A =
 tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [12., 13., 14., 15.],
        [16., 17., 18., 19.]])
Ax = tensor([ 14.,  38.,  62.,  86., 110.])
AB =
 tensor([[ 6.,  6.,  6.],
        [22., 22., 22.],
        [38., 38., 38.],
        [54., 54., 54.],
        [70., 70., 70.]])


### 2.7 范数（Norm）

范数是衡量向量"大小"的函数，满足：非负性、三角不等式、齐次性。

| 范数 | 公式 | 代码 | 特点 |
|------|------|------|------|
| $L_1$ 范数 | $\|\mathbf{x}\|_1 = \sum\|x_i\|$ | `torch.abs(x).sum()` | 对异常值不敏感 |
| $L_2$ 范数 | $\|\mathbf{x}\|_2 = \sqrt{\sum x_i^2}$ | `torch.norm(x)` | 最常用（欧几里得距离） |
| Frobenius 范数 | $\|\mathbf{X}\|_F = \sqrt{\sum x_{ij}^2}$ | `torch.norm(X)` | 矩阵的$L_2$范数 |

深度学习中，目标函数常表达为范数形式（如最小化预测与真实值的距离）。

## 三、微积分（Calculus）

深度学习的核心是**优化**：最小化损失函数。微积分为此提供数学基础。

### 3.1 导数（Derivative）

导数是函数相对于其变量的**瞬时变化率**，也是曲线切线的斜率。

$$f'(x) = \lim_{h \rightarrow 0} \frac{f(x+h) - f(x)}{h}$$

**常用求导规则：**
- $DC = 0$（常数导数为0）
- $Dx^n = nx^{n-1}$（幂律）
- $De^x = e^x$，$D\ln(x) = 1/x$
- 加法法则：$(f+g)' = f' + g'$
- 乘法法则：$(fg)' = f'g + fg'$
- 链式法则：$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$

In [22]:
import numpy as np

def f(x):
    return 3 * x ** 2 - 4 * x

def numerical_lim(f, x, h):
    return (f(x + h) - f(x)) / h

# 数值验证：f'(1) = 6*1 - 4 = 2
h = 0.1
for i in range(5):
    print(f'h={h:.5f}, 数值导数={numerical_lim(f, 1, h):.5f}')
    h *= 0.1

h=0.10000, 数值导数=2.30000
h=0.01000, 数值导数=2.03000
h=0.00100, 数值导数=2.00300
h=0.00010, 数值导数=2.00030
h=0.00001, 数值导数=2.00003


### 3.2 偏导数（Partial Derivative）

多元函数$y = f(x_1, \ldots, x_n)$关于$x_i$的偏导数：

$$\frac{\partial y}{\partial x_i} = \lim_{h \rightarrow 0} \frac{f(x_1, \ldots, x_i+h, \ldots, x_n) - f(x_1, \ldots, x_n)}{h}$$

将其他变量视为常数，只对$x_i$求导。

### 3.3 梯度（Gradient）

梯度是偏导数组成的**向量**：

$$\nabla_{\mathbf{x}} f(\mathbf{x}) = \left[\frac{\partial f}{\partial x_1}, \frac{\partial f}{\partial x_2}, \ldots, \frac{\partial f}{\partial x_n}\right]^\top$$

梯度指向函数值**增长最快**的方向，反方向是**下降最快**的方向（梯度下降的基础）。

**常用梯度规则：**
- $\nabla_{\mathbf{x}} \mathbf{A}\mathbf{x} = \mathbf{A}^\top$
- $\nabla_{\mathbf{x}} \|\mathbf{x}\|^2 = 2\mathbf{x}$

### 3.4 链式法则（Chain Rule）

复合函数$y = f(g(x))$的求导：

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$$

这是**反向传播（backpropagation）** 的数学基础。

## 四、自动微分（Automatic Differentiation）

手工对复杂模型求导容易出错。深度学习框架通过**自动微分**自动计算导数。

核心机制：构建**计算图（computational graph）**，跟踪数据通过哪些操作产生输出，然后**反向传播（backpropagate）** 梯度。意味着跟踪整个计算图，填充关于每个参数的偏导数。

### 4.1 基本用法

关键步骤：
1. `x.requires_grad_(True)` — 标记需要求导的变量
2. 前向计算 `y = f(x)`
3. 反向传播 `y.backward()`
4. 访问梯度 `x.grad`

**注意**：PyTorch默认**累积梯度**，每次反向传播前需 `x.grad.zero_()`

In [1]:
import torch

# 示例1：y = 2x^Tx，梯度应为 4x
x = torch.arange(4.0, requires_grad=True)  # 标记需要求导
y = 2 * torch.dot(x, x)                     # y = 2*(0+1+4+9) = 28
y.backward()                                 # 反向传播
print("y = 2x^Tx 的梯度:", x.grad)           # [0, 4, 8, 12] = 4x
print("验证:", x.grad == 4 * x)

# 清除梯度（重要！）
x.grad.zero_()
y = x.sum()   # y = x1+x2+x3+x4
y.backward()
print("y = sum(x) 的梯度:", x.grad)  # [1, 1, 1, 1]

y = 2x^Tx 的梯度: tensor([ 0.,  4.,  8., 12.])
验证: tensor([True, True, True, True])
y = sum(x) 的梯度: tensor([1., 1., 1., 1.])


### 4.2 非标量反向传播

当`y`不是标量时，通常调用 `y.sum().backward()` 来计算各分量偏导数之和。

### 4.3 分离计算（Detach）

将某些计算移动到记录的计算图之外。切断张量梯度传播链路，让部分计算不参与反向传播、不更新参数。

`u = y.detach()` 将`y`从计算图中分离，`u`的值与`y`相同，但梯度不会流经`u`。

二、核心作用

阻断梯度回传，只做前向计算，不求梯度

固定部分参数 / 中间结果，避免被训练更新

节省计算、内存，防止梯度意外传递

三、常用场景（深度学习 / 反向传播）

冻结网络层：只训练部分网络，另一部分权重固定

损失分支分离：某一分支结果只用作计算，不反向传梯度

循环结构 / 自监督任务：阻止梯度循环传递、梯度爆炸

提取特征：拿中间特征做后续运算，但不影响原网络梯度

四、关键特点

前向传播：数值完全不变

反向传播：梯度到此为止，不再向前传递

被分离的张量 / 层：梯度为 0，参数不更新

### 4.4 Python控制流的梯度

自动微分即使在函数包含 `if`/`while` 等Python控制流时也能正确计算梯度。

In [ ]:
# 分离计算示例
# 有点类似用一个新变量u代替旧变量值y，保留y的计算图，但u不参与反向传播
x = torch.arange(4.0, requires_grad=True)
y = x * x          # y = x^2
u = y.detach()     # 分离y
z = u * x          # z = u*x（u被视为常数）

z.sum().backward()
print("z = detach(x^2) * x 的梯度:", x.grad)  # = u = [0, 1, 4, 9]

# 验证：再次对y求导
x.grad.zero_()
y.sum().backward()
print("y = x^2 的梯度:", x.grad)  # = 2x = [0, 2, 4, 6]

z = detach(x^2) * x 的梯度: tensor([0., 1., 4., 9.])
y = x^2 的梯度: tensor([0., 2., 4., 6.])


## 五、概率（Probability）

机器学习的本质是做出**预测**，而预测往往涉及不确定性。概率提供了量化不确定性的语言。

### 5.1 基本概率论

- **样本空间（sample space）** $\mathcal{S}$：所有可能结果的集合
- **事件（event）**：样本空间的子集
- **概率（probability）** $P(\mathcal{A})$：事件发生的可能性，满足 $0 \leq P \leq 1$

**大数定律（law of large numbers）**：随着试验次数增加，相对频率收敛到真实概率。

In [3]:
import torch
from torch.distributions import multinomial

# 模拟掷骰子：大数定律
fair_probs = torch.ones([6]) / 6  # 公平骰子，每个面概率1/6

# 单次采样
print("单次采样:", multinomial.Multinomial(1, fair_probs).sample())

# 1000次采样 → 相对频率 ≈ 真实概率(1/6 ≈ 0.167)
counts = multinomial.Multinomial(1000, fair_probs).sample()
print("1000次投掷的相对频率:", counts / 1000)

单次采样: tensor([0., 0., 0., 0., 0., 1.])
1000次投掷的相对频率: tensor([0.1710, 0.1660, 0.1630, 0.1580, 0.1640, 0.1780])


### 5.2 处理多个随机变量

| 概念 | 英文 | 公式 | 含义 |
|------|------|------|------|
| 联合概率 | joint probability | $P(A=a, B=b)$ | A和B同时发生的概率 |
| 条件概率 | conditional probability | $P(B \mid A)$ | 已知A发生，B发生的概率 |
| 边际化 | marginalization | $P(B) = \sum_A P(A, B)$ | 对所有A求和得到B的概率 |
| 独立性 | independence | $P(A,B) = P(A)P(B)$ | A和B互不影响 |

### 5.3 贝叶斯定理（Bayes' Theorem）

$$P(A \mid B) = \frac{P(B \mid A) \cdot P(A)}{P(B)}$$

**核心思想**：用观测结果$B$更新对原因$A$的信念。
- $P(A)$ — **先验概率**（prior）：观测前的信念
- $P(A \mid B)$ — **后验概率**（posterior）：观测后的信念
- $P(B \mid A)$ — **似然**（likelihood）：原因$A$下观测到$B$的可能性

**应用**：医学诊断、垃圾邮件过滤、分类器设计。

### 5.4 期望和方差

| 概念 | 英文 | 公式 | 含义 |
|------|------|------|------|
| 期望 | expectation | $E[X] = \sum_x x P(X=x)$ | 随机变量的平均值 |
| 方差 | variance | $\mathrm{Var}[X] = E[(X - E[X])^2] = E[X^2] - E[X]^2$ | 衡量偏离均值的程度 |
| 标准差 | standard deviation | $\sqrt{\mathrm{Var}[X]}$ | 方差的平方根 |

## 六、本章核心知识总结

### 关键术语速查表

| 术语 | 英文 | 核心含义 |
|------|------|----------|
| 张量 | tensor | $n$维数组，深度学习的数据载体 |
| 广播 | broadcasting | 不同形状张量的按元素运算 |
| 点积 | dot product | $\mathbf{x}^\top\mathbf{y} = \sum x_i y_i$ |
| Hadamard积 | Hadamard product | 矩阵按元素乘法 $A \odot B$ |
| 矩阵乘法 | matrix multiplication | $c_{ij} = \mathbf{a}^\top_i \mathbf{b}_j$ |
| 范数 | norm | 衡量向量/矩阵的"大小" |
| 导数 | derivative | 函数的瞬时变化率 |
| 梯度 | gradient | 偏导数组成的向量 |
| 链式法则 | chain rule | 复合函数求导法则 |
| 自动微分 | autograd | 框架自动计算梯度 |
| 反向传播 | backpropagation | 沿计算图反向填充梯度 |
| 条件概率 | conditional probability | $P(B \mid A)$ |
| 贝叶斯定理 | Bayes' theorem | $P(A \mid B) = \frac{P(B \mid A)P(A)}{P(B)}$ |
| 期望 | expectation | 随机变量的平均值 |
| 方差 | variance | 偏离均值的程度 |

### 代码速查


In [ ]:
# 创建张量
torch.arange(12).reshape(3, 4)  # 连续整数
torch.zeros((2, 3))              # 全0
torch.randn(3, 4)                # 正态随机

# 运算
torch.dot(x, y)     # 点积
torch.mv(A, x)      # 矩阵-向量积
torch.mm(A, B)      # 矩阵乘法
torch.norm(x)       # L2范数
torch.cat((X, Y), dim=0)  # 连结

# 自动微分
x = torch.tensor([1.0, 2, 3], requires_grad=True)
y = (x ** 2).sum()
y.backward()
x.grad  # 获取梯度